In [1]:
import os
import numpy as np
import pandas as pd
from natsort import natsorted
from google.colab import drive
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error


DATASET_PATH = 'Datasets'

MODEL_NAME = 'TraditionalSRGM_STVR'
OUTPUT_PATH = f'Results/{MODEL_NAME}'
EXCEL_FILE_NAME = "Total_Results_TraditionalSRGM_BEST(6).xlsx"

def goel_okumoto_model(x, a, b):
    return a * (1 - np.exp(-b * x))

def yamada_delayed_s_shaped_model(x, a, b):
    return a * (1 - (1 + b * x) * np.exp(-b * x))

def inflection_s_shaped_model(x, a, b, c):
    return a * (1 - np.exp(-b * x)) / (1 + ((1 - c) / c) * np.exp(-b * x))

def generalized_goel_okumoto_model(x, a, b, c):
    return a * (1 - np.exp(-b * (x ** c)))

def logistic_model(x, a, b, c):
    return a / (1 + (c * np.exp(-b * x)))

def gompertz_model(x, a, b, c):
    return a * (b ** (c ** x))

models = {
    "Goel-Okumoto": {"func": goel_okumoto_model, "bounds": ([0, 0], [np.inf, 1])},
    "Yamada Delayed S-Shaped": {"func": yamada_delayed_s_shaped_model, "bounds": ([0, 0], [np.inf, 1])},
    "Inflection S-Shaped": {"func": inflection_s_shaped_model, "bounds": ([0, 0, 0], [np.inf, 1, 1])},
    "Generalized Goel-Okumoto": {"func": generalized_goel_okumoto_model, "bounds": ([0, 0, 0], [np.inf, 1, np.inf])},
    "Logistic": {"func": logistic_model, "bounds": ([0, 0, 0], [np.inf, np.inf, np.inf])},
    "Gompertz": {"func": gompertz_model, "bounds": ([0, 0, 0], [np.inf, 1, 1])},
}

def main():
    # drive.mount('/content/drive')
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    results = []
    csv_files = [file for file in natsorted(os.listdir(DATASET_PATH)) if file.endswith('.csv')]

    for file_name in csv_files:
        data = pd.read_csv(os.path.join(DATASET_PATH, file_name))
        time = data['time'].to_numpy()
        failure_count = data['cumulative_failure_count'].to_numpy()

        split_index = int(len(time) * 0.5)
        time_input, time_output = time[:split_index], time[split_index:]
        failure_count_input, failure_count_output = failure_count[:split_index], failure_count[split_index:]

        best_model = None
        best_mse = float('inf')
        best_params = None

        for model_name, model_info in models.items():
            try:
                params, _ = curve_fit(model_info["func"], time_input, failure_count_input, bounds=model_info["bounds"])
                failure_count_input_pred = model_info["func"](time_input, *params)
                mse = mean_squared_error(failure_count_input, failure_count_input_pred)
                rmse = np.sqrt(mse)
                range_input = np.max(failure_count_input) - np.min(failure_count_input)
                input_nrmse = rmse / range_input if range_input != 0 else 0

                if mse < best_mse:
                    best_mse = mse
                    best_model = model_name
                    best_params = params

            except Exception as e:
                continue

        if best_model and best_params is not None:
            failure_count_output_pred = models[best_model]["func"](time_output, *best_params)

            metrics = {
                "Dataset": file_name[:-4],
                "RMSE": np.sqrt(mean_squared_error(failure_count_output, failure_count_output_pred)),
                "MAE": mean_absolute_error(failure_count_output, failure_count_output_pred),
                "MAPE": 100 * mean_absolute_percentage_error(failure_count_output, failure_count_output_pred),
                "PP": np.mean(np.square(failure_count_output - failure_count_output_pred) / failure_count_output),
                "PRR": np.mean(np.square(failure_count_output - failure_count_output_pred) / failure_count_output_pred),
                "Best Model": best_model,
                "Input MSE": best_mse,
                "Input nRMSE": input_nrmse,
                "param_a": best_params[0],
                "param_b": best_params[1] if len(best_params) > 1 else None,
                "param_c": best_params[2] if len(best_params) > 2 else None,
                "param_d": best_params[3] if len(best_params) > 3 else None,
                "param_e": best_params[4] if len(best_params) > 4 else None,
            }
            results.append(metrics)

        else:
            print(f"Could not determine a best model for {file_name}")
            metrics = {
                "Dataset": file_name[:-4],
                "RMSE": float('inf'),
                "MAE": float('inf'),
                "MAPE": float('inf'),
                "PP": float('inf'),
                "PRR": float('inf'),
                "Best Model": None,
            }

    results_df = pd.DataFrame(results)
    results_df.to_excel(os.path.join(OUTPUT_PATH, EXCEL_FILE_NAME), index=False)

if __name__ == "__main__":
    main()
